# GIU portal data as agent tools

This notebook turns the verified GIU portal workflow into **LangChain tools**.
The tools stay read-only: they inspect dropdowns and grade/transcript pages but
do not register courses or submit university forms.

**CMS is intentionally out of scope.** This version does not import, configure,
or call any CMS functionality.

The important detailed-grade flow is the same one you use in the browser:

1. `list_grade_seasons` reads the Season dropdown.
2. `list_courses_in_season` selects a season and reads its Course dropdown.
3. `get_course_grades` selects a course and returns assessments and midterm results.

A course may have a long label such as
`GIU-Cairo. Informatic and Computer Science Thesis & Internship - INCS700 Bachelor Thesis`.
The tools accept either that full label or a unique fragment such as `INCS700`.

GIU can throttle requests. Results are cached, and retryable requests wait before
trying again, so a tool may take about a minute when the portal is busy.

For the advisory demo, we treat **Winter 2024** as the current semester and
**2024-2025** as its transcript year. These are explicit assumptions, not values
read from GIU's live-current-course page.

## 1. Load the GIU account from `.env`

Keep `.env` beside this notebook. It must contain `GUC_USERNAME`, `GUC_PASSWORD`,
`GUC_SITE=giu`, and `ANTHROPIC_API_KEY`. The cell checks names only and never
prints secret values.

In [8]:
import os
import importlib
from pathlib import Path
from dotenv import load_dotenv

ENV_PATH = Path.cwd() / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(f"Expected credentials at {ENV_PATH}")
load_dotenv(ENV_PATH, override=True)

required = ["GUC_USERNAME", "GUC_PASSWORD", "GUC_SITE"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Add these variables to .env: {', '.join(missing)}")
if os.environ["GUC_SITE"].casefold() != "giu":
    raise RuntimeError("Set GUC_SITE=giu in .env for this notebook.")

# Reload local modules so source fixes are picked up after rerunning this cell.
import guc_portal._sites as portal_sites_module
import guc_portal.client as portal_client_module

importlib.reload(portal_sites_module)
importlib.reload(portal_client_module)
GucPortal = portal_client_module.GucPortal

portal = GucPortal(site="giu")
print("Portal client ready (credentials hidden).")
print("Site:", portal.site_name)
print("Base URL:", portal.base_url)

Portal client ready (credentials hidden).
Site: giu
Base URL: https://portal.giu-uni.de/GIUb


## 2. Build the read-only tools

The two constants below define the simulated advisory context. Change them later
in one place if you want the agent to focus on another semester or transcript year.

The private helpers cache GIU results and resolve readable labels to portal values.
Matching is case-insensitive. An exact label wins; otherwise the query must match
exactly one option, so a vague request cannot silently select the wrong course.

In [9]:
from langchain.tools import tool

ADVISORY_YEAR = "2024-2025"
CURRENT_SEASON = "Winter 2024"  # simulated current semester

_cache = {}


def _seasons():
    if "seasons" not in _cache:
        _cache["seasons"] = portal.available_seasons(tries=2, delay=60)
    return _cache["seasons"]


def _courses(season_value: str):
    key = ("courses", season_value)
    if key not in _cache:
        _cache[key] = portal.list_previous_courses(
            season_value, tries=2, delay=60
        )
    return _cache[key]


def _years():
    if "years" not in _cache:
        _cache["years"] = portal.available_years(tries=2, delay=60)
    return _cache["years"]


def _pick(options, query: str, kind: str):
    """Return one (value, label), rejecting missing or ambiguous text."""
    needle = query.strip().casefold()
    exact = [(value, label) for value, label in options if label.casefold() == needle]
    matches = exact or [
        (value, label) for value, label in options if needle in label.casefold()
    ]
    if not matches:
        raise ValueError(f"No {kind} matches {query!r}.")
    if len(matches) > 1:
        labels = [label for _value, label in matches]
        raise ValueError(f"Ambiguous {kind} {query!r}; matches: {labels}")
    return matches[0]


def _transcript(year_value: str):
    key = ("transcript", year_value)
    if key not in _cache:
        _cache[key] = portal.get_transcript_year(year_value)
    return _cache[key]


@tool
def list_grade_seasons() -> list[str]:
    """List GIU seasons that have detailed grades, e.g. 'Winter 2025'."""
    return [label for _v, label in _seasons()]


@tool
def list_courses_in_season(season: str) -> dict:
    """List exact GIU course labels inside one grade season.
    `season` is a label such as 'Winter 2025'. Call list_grade_seasons first
    when the user did not provide a valid season."""
    try:
        season_value, season_label = _pick(_seasons(), season, "season")
        return {
            "season": season_label,
            "courses": [label for _value, label in _courses(season_value)],
        }
    except ValueError as exc:
        return {"error": str(exc), "available_seasons": [l for _v, l in _seasons()]}
    except Exception as exc:
        return {"error": str(exc), "advice": "GIU may be busy; retry in about one minute."}


@tool
def get_course_grades(season: str, course: str) -> dict:
    """Get detailed grades for one course in one past GIU season.
    `course` may be its full dropdown label or a unique fragment such as
    'INCS700'. Returns assessment elements and the midterm-results table."""
    try:
        season_value, season_label = _pick(_seasons(), season, "season")
        course_value, course_label = _pick(
            _courses(season_value), course, "course"
        )
        key = ("grades", season_value, course_value)
        if key not in _cache:
            grades = portal.get_previous_grades(
                season_value, course_value, tries=2, delay=60
            )
            _cache[key] = {
                "season": season_label,
                "course": course_label,
                "assessments": [
                    {
                        "assessment": item.assessment,
                        "element": item.element,
                        "grade": item.grade,
                        "evaluator": item.evaluator,
                    }
                    for item in grades.items
                ],
                "midterm_results": grades.percentages,
            }
        return _cache[key]
    except Exception as exc:
        return {"error": str(exc)}


@tool
def list_transcript_years() -> list[str]:
    """List the academic years on the transcript, e.g. '2024-2025'."""
    return [label for _v, label in _years()]


@tool
def get_transcript(year: str) -> dict:
    """Get one GIU transcript year with courses, grades, credits, and GPA.
    `year` is an academic-year label such as '2024-2025'."""
    try:
        year_value, year_label = _pick(_years(), year, "transcript year")
        transcript = _transcript(year_value)
        return {
            "year": year_label,
            "cumulative_gpa": transcript.cumulative_gpa,
            "courses": [
                {
                    "semester": row.semester,
                    "course": row.course,
                    "grade": row.grade,
                    "numeric": row.numeric,
                    "hours": row.hours,
                    "group": row.group,
                }
                for row in transcript.rows
            ],
        }
    except ValueError as exc:
        return {"error": str(exc), "available_years": [l for _v, l in _years()]}
    except Exception as exc:
        return {"error": str(exc), "advice": "GIU may be busy; retry in about one minute."}


@tool
def find_transcript_course(year: str, course: str) -> dict:
    """Find transcript rows by course name/code within one academic year.
    This searches a cached transcript locally after it is fetched."""
    data = get_transcript.invoke({"year": year})
    if "error" in data:
        return data
    needle = course.strip().casefold()
    matches = [row for row in data["courses"] if needle in row["course"].casefold()]
    return {"year": data["year"], "query": course, "matches": matches}


@tool
def get_advisory_context() -> dict:
    """Return the configured academic-advisory assumptions and data limits."""
    return {
        "simulated_current_season": CURRENT_SEASON,
        "transcript_year": ADVISORY_YEAR,
        "data_sources": ["GIU portal detailed grades", "GIU transcript"],
        "excluded_sources": ["CMS", "live current-course page"],
    }


@tool
def list_advisory_courses() -> dict:
    """List courses in the simulated current semester configured by the notebook."""
    return list_courses_in_season.invoke({"season": CURRENT_SEASON})


@tool
def get_advisory_course_grades(course: str) -> dict:
    """Get grades for one course in the simulated current semester.
    `course` may be its exact GIU label or a unique code/name fragment."""
    return get_course_grades.invoke({"season": CURRENT_SEASON, "course": course})


@tool
def get_advisory_transcript() -> dict:
    """Get the transcript for the configured advisory academic year."""
    return get_transcript.invoke({"year": ADVISORY_YEAR})


@tool
def find_advisory_transcript_course(course: str) -> dict:
    """Find a course in the configured advisory-year transcript."""
    return find_transcript_course.invoke({"year": ADVISORY_YEAR, "course": course})

## 3. Inspect the tool belt before adding a model

This cell does not contact GIU. It only collects the tools and prints their names,
which is a quick way to confirm what the future agent will be allowed to call.

In [10]:
portal_tools = [
    get_advisory_context,
    list_advisory_courses,
    get_advisory_course_grades,
    get_advisory_transcript,
    find_advisory_transcript_course,
    list_grade_seasons,
    list_courses_in_season,
    get_course_grades,
    list_transcript_years,
    get_transcript,
    find_transcript_course,
]

print("Agent tool belt:")
for portal_tool in portal_tools:
    print(" -", portal_tool.name)

Agent tool belt:
 - get_advisory_context
 - list_advisory_courses
 - get_advisory_course_grades
 - get_advisory_transcript
 - find_advisory_transcript_course
 - list_grade_seasons
 - list_courses_in_season
 - get_course_grades
 - list_transcript_years
 - get_transcript
 - find_transcript_course


## 4. Build the GIU portal agent

The model receives tool descriptions, not your credentials. The Python tool calls
use the already authenticated `portal` object. The prompt teaches the agent the
verified dropdown order, the simulated-current-semester assumption, and the limits
of advice that can be produced without CMS or degree-requirement data.

In [11]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("Add ANTHROPIC_API_KEY to the same .env file.")

model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

agent = create_agent(
    model=model,
    tools=portal_tools,
    system_prompt=(
        f"You are a read-only academic advisor and study-planning assistant for a GIU student. "
        "Use portal tools for every factual claim about the student's data. "
        f"Treat {CURRENT_SEASON} as the simulated current semester and {ADVISORY_YEAR} "
        "as its advisory transcript year. When the user says 'current semester' or "
        "'my courses', use the advisory tools rather than GIU's live-current page. "
        "For past detailed grades, follow GIU's order: list seasons, select one, "
        "list its courses when needed, then fetch one course's grades. A course "
        "code such as INCS700 may identify its long dropdown label. "
        "For transcript questions, choose an academic year before fetching it. "
        "You may summarize performance, identify strong and weak assessments, compare "
        "available grades, suggest study priorities, and prepare questions for an official advisor. "
        "CMS is unavailable and out of scope. You do not know course content, deadlines, "
        "attendance, prerequisites, schedules, or graduation requirements unless the user "
        "provides them. Clearly separate portal facts from your recommendations. "
        "Do not present recommendations as official university policy. "
        "The portal is slow: make the fewest calls possible and reuse tool results. "
        "Do not fetch every course's detailed grades unless the user explicitly requests it. "
        "Never invent a grade, GPA, season, or course. Explain empty results plainly."
    ),
)


conversation = []


def ask(question: str):
    """Ask a question while continuing the same advisory conversation."""
    start = len(conversation)
    result = agent.invoke({"messages": [*conversation, HumanMessage(question)]})
    conversation[:] = result["messages"]
    result["_new_message_start"] = start
    return result


def reset_conversation():
    """Start a fresh advisory conversation; the portal response cache remains."""
    conversation.clear()
    print("Conversation cleared.")


def pretty(result):
    """Print the human, agent, and tool messages so the workflow is visible."""
    start = result.get("_new_message_start", 0)
    for message in result["messages"][start:]:
        body = message.content if message.content else getattr(message, "tool_calls", "")
        print(message.type.upper().ljust(6), "→", body)

## 5. Try the agent and watch its tool calls

Run the cell below first. It should state the hard-coded assumption and list the
courses from Winter 2024's historical dropdown—not the empty live-current page.

Then reuse `ask(...)` for follow-ups. The conversation continues automatically:

- **Course discovery:** `List my courses in the current advisory semester.`
- **One-course diagnosis:** `Analyze ICS501. Which assessment needs the most attention?`
- **Study planning:** `Make a one-week improvement plan based only on those grades.`
- **Transcript review:** `Summarize my performance in the advisory year and identify patterns.`
- **Advisor preparation:** `Prepare a short evidence-based brief for my advisor meeting.`
- **Past semester:** `List my Spring 2024 courses and inspect <course code>.`

Use `reset_conversation()` when you want a fresh discussion. The printed `TOOL`
rows prove which portal function supplied each fact.

With portal data alone, the agent can analyze grades, GPA, course history, weak
assessments, and study priorities. It cannot verify prerequisites, course material,
deadlines, schedules, or graduation eligibility because CMS and official program
requirements are not connected. You can still paste a requirement or policy into
the conversation and ask it to compare that text with your transcript.

In [12]:
question = "What semester are we treating as current, and what courses are in it? List the grades of ICS501 and a plan to improve them"
result = ask(question)
pretty(result)

HUMAN  → What semester are we treating as current, and what courses are in it? List the grades of ICS501 and a plan to improve them
AI     → [{'text': "I'll start by checking the advisory context and listing the current courses.", 'type': 'text'}, {'id': 'toolu_01XYw1gGYUUU4ntMxLSuiFjj', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'get_advisory_context', 'type': 'tool_use'}, {'id': 'toolu_01V8ueUTC7ehBWZyeyEXeZH3', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'list_advisory_courses', 'type': 'tool_use'}]
TOOL   → {"simulated_current_season": "Winter 2024", "transcript_year": "2024-2025", "data_sources": ["GIU portal detailed grades", "GIU transcript"], "excluded_sources": ["CMS", "live current-course page"]}
TOOL   → {"season": "Winter 2024", "courses": ["GIU-Cairo.Informatics and Computer Science 3rd - INCS102 Operating Systems", "GIU-Cairo.Informatics and Computer Science - Software Engineering 5th - ICS501 Software Project I", "GIU-Cairo.Informatics and Computer Science